<a href="https://colab.research.google.com/github/margobergman-now/pdc_evergreen_git/blob/main/2D_grid_of_2D_blocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%writefile block.cu

#include <stdio.h>

 // CUDA runtime
 #include <cuda_runtime.h>

 __global__ void hello() {
     // special dim3 variables available to each thread in a kernel
     // or device function:
     // blockIdx    the x, y, z coordinate of the block in the grid
     // threadIdX   the x, y, z coordinate of the thread in the block
     printf("I am thread (%d, %d, %d) of block (%d, %d, %d) in the grid\n",
         threadIdx.x, threadIdx.y, threadIdx.z,
         blockIdx.x, blockIdx.y, blockIdx.z );

 }

 // Note that this is called from the host, not the GPU device.
 // We create dim3 structs there and can print their components
 // with this function
 void printDims(dim3 gridDim, dim3 blockDim) {
     printf("Grid Dimensions : [%d, %d, %d] blocks. \n",
     gridDim.x, gridDim.y, gridDim.z);

     printf("Block Dimensions : [%d, %d, %d] threads.\n",
     blockDim.x, blockDim.y, blockDim.z);
 }

 int main(int argc, char **argv) {

     // dim3 is a special data type: a vector of 3 integers.
     // each integer is accessed using .x, .y and .z (see printDims() above)

     // 2D dimensionsional case is the following:
     // 2D grid of 2D blocks
     dim3 gridDim(2,2);     // 2 blocks in x, y direction, z defaults to 1
     dim3 blockDim(2,2);    // 4 threads per block: 2 in x direction, 2 in y
     // TODO: change to 8 threads per block: 4 in x direction, 2 in y

     printDims(gridDim, blockDim);

     printf("From each thread:\n");
     hello<<<gridDim, blockDim>>>();
     cudaDeviceSynchronize();      // need for printfs in kernel to flush

     return 0;
 }


Writing block.cu


In [6]:
%%shell
nvcc block.cu -o block

In [7]:
%%shell

./block

Grid Dimensions : [2, 2, 1] blocks. 
Block Dimensions : [2, 2, 1] threads.
From each thread:


In [1]:
!nvidia-smi

Mon Nov 17 18:37:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
%%writefile block_2.cu

 //
 // Demonstrate one way of mapping threads in a 2D grid of 2D blocks
 // to indexes in a 2D array of data values.
 //

 #include <stdio.h>
 #include <cuda_runtime.h>

 ///////
 // error checking macro taken from Oakridge Nat'l lab training code:
 // https://github.com/olcf/cuda-training-series
 ////////
 #define cudaCheckErrors(msg) \
     do { \
         cudaError_t __err = cudaGetLastError(); \
         if (__err != cudaSuccess) { \
             fprintf(stderr, "Fatal error: %s (%s at %s:%d)\n", \
                 msg, cudaGetErrorString(__err), \
                 __FILE__, __LINE__); \
             fprintf(stderr, "*** FAILED - ABORTING\n"); \
             exit(1); \
         } \
     } while (0)

 // print information about the mapping of a thread in a 2D block
 // where the blocks are in a 2D grid
 __global__ void find2DIndex() {
     int row = (blockIdx.y * blockDim.y) + threadIdx.y;
     int column = (blockIdx.x * blockDim.x) + threadIdx.x;

     printf("block (%d, %d) thread (%d, %d) maps to (%d, %d) of array\n",
         blockIdx.x, blockIdx.y,
         threadIdx.x, threadIdx.y,
         row, column);
 }

 int main(int argc, char **argv) {

     printf("Host calls: 2x2 grid of 2x2 blocks of 4 threads each\n");
     printf("2D grid of blocks\n");

     // Kernel configuration, where a two-dimensional grid and
     // two-dimensional blocks are configured.
     dim3 dimGrid2D(2, 2);                      // 2x2 = 4 blocks
     dim3 dimBlock2D(2, 2);                  // 2x2 = 4 threads per block
     // TODO: try replacing the above line with the following
     // by commenting and uncommenting and re-compiling:
     //   dim3 dimBlock2D(512, 2);
     // TODO: then try this one next:
     //   dim3 dimBlock2D(1024, 2);

     find2DIndex<<<dimGrid2D, dimBlock2D>>>();
     cudaCheckErrors("kernel launch failure");

     cudaDeviceSynchronize();
     cudaCheckErrors("Failure to synchronize device");

     return 0;
 }


Writing block_2.cu


In [11]:
%%shell
nvcc -arch=sm_75 block_2.cu -o block_2

In [12]:
%%shell

./block_2

Host calls: 2x2 grid of 2x2 blocks of 4 threads each
2D grid of blocks
block (0, 1) thread (0, 0) maps to (2, 0) of array
block (0, 1) thread (1, 0) maps to (2, 1) of array
block (0, 1) thread (0, 1) maps to (3, 0) of array
block (0, 1) thread (1, 1) maps to (3, 1) of array
block (0, 0) thread (0, 0) maps to (0, 0) of array
block (0, 0) thread (1, 0) maps to (0, 1) of array
block (0, 0) thread (0, 1) maps to (1, 0) of array
block (0, 0) thread (1, 1) maps to (1, 1) of array
block (1, 1) thread (0, 0) maps to (2, 2) of array
block (1, 1) thread (1, 0) maps to (2, 3) of array
block (1, 1) thread (0, 1) maps to (3, 2) of array
block (1, 1) thread (1, 1) maps to (3, 3) of array
block (1, 0) thread (0, 0) maps to (0, 2) of array
block (1, 0) thread (1, 0) maps to (0, 3) of array
block (1, 0) thread (0, 1) maps to (1, 2) of array
block (1, 0) thread (1, 1) maps to (1, 3) of array


In [23]:
%%writefile seq_block.cu

 /*
  * Sequential version of matrix multiplication.
  */
 #include <stdio.h>
 #include <stdlib.h>
 #include <math.h>
 #include <omp.h> // just for timing


// fill a given square matrix with rows of float values
// equal to each row number
void fillMatrix(int size, float * A) {
  for (int i = 0; i < size; ++i) {
      for (int j = 0; j < size; ++j) {
        A[i*size + j] = ((float)i);
      }
  }
}

// display a given square matrix for debugging purposes
void showMatrix(int size, float * matrix) {
  int i, j;
  for (i=0; i<size; i++){
      for (j=0; j<size; j++) {
          printf("element [%d][%d] = %f \n",
                  i,j, matrix[i*size + j]);
      }
  }
}

void debugPrintMatrix(int verbose, int size,
                      float *matrix, const char *msg) {
  if (verbose){
    printf("%s \n", msg);
    showMatrix(size, matrix);
  }
}

// Check whether last row of result matrix is what we expect
void verifyCorrect(int size, float *matrix) {
  // determine what the last row should contain
  float lastRowValue = 0.0;
  float maxError = 0.0;
  float nextVal = 0.0;

  for (int i=0; i<size; i++)
      lastRowValue += i * (size-1);

  for (int j=0; j<size; j++) {
    nextVal = matrix[(size-1)*size + j];
    maxError = fmaxf(maxError, fabs(nextVal - lastRowValue));
  }
  printf("max error of last row matrix C values: %f\n",
          maxError);
}


 // function declarations
 void MatrixMult(int size, float *__restrict__ A,
                 float *__restrict__ B,
                 float *__restrict__ C); //
 void getArguments(int argc, char **argv,
                   int *size, int *verbose);

 int main(int argc, char **argv) {

   // default values
   int size = 256;     // num rows, cols of square matrix
   int verbose = 0;    // default to not printing matrices
   //change defaults if arguments given
   getArguments(argc, argv, &size, &verbose);

   printf("matrix rows, cols = %d\n", size);

   float *A; // input matrix
   float *B; // input matrix
   float *C; // output matrix

   // Use a 'flattened' 1D array of contiguous
   // memory for the matrices
   // size = number of rows = number of columns
   // in the square matrices
   size_t num_elements = size * size * sizeof(float);
   A = (float *)malloc(num_elements);
   B = (float *)malloc(num_elements);
   C = (float *)malloc(num_elements);

   fillMatrix(size, A);
   fillMatrix(size, B);
   char msgA[32] = "matrix A after filling:";
   debugPrintMatrix(verbose, size, A, msgA);

   double startTime = omp_get_wtime();

   MatrixMult(size, A, B, C);

   char msgC[32] = "matrix C after MatrixMult(): ";
   debugPrintMatrix(verbose, size, C, msgC);

   double endTime = omp_get_wtime();

   printf("\nTotal runtime %f seconds (%f milliseconds)\n",
         (endTime - startTime), (endTime - startTime) * 1000);

   verifyCorrect(size, C);

   free(A);
   free(B);
   free(C);
   return 0;
 }
 ////////////////////////////////////// end main

 // mutiply matrix A times matrix B, placing result in matrix C
 void MatrixMult(int size, float *__restrict__ A,
                 float *__restrict__ B, float *__restrict__ C)
 {

   for (int i = 0; i < size; ++i)
   {
     for (int j = 0; j < size; ++j)
     {
       float tmp = 0.;
       for (int k = 0; k < size; ++k)
       {
         tmp += A[i * size + k] * B[k * size + i];
       }
       C[i * size + j] = tmp; // update cell of C once
     }
   }
 }

 void getArguments(int argc, char **argv,
                   int *size, int *verbose) {
   // 2 arguments optional:
   //   size of one side of square matrix
   //   verbose printing for debugging
   if (argc > 3)
   {
     fprintf(stderr, "Use: %s [size] [verbose]\n", argv[0]);
     exit(EXIT_FAILURE);
   }

   if (argc >= 2)
   {
     *size = atoi(argv[1]);
     if (argc == 3)
     {
       *verbose = atoi(argv[2]);
     }
   }

   if (*verbose)
   {
     printf("size of matrix side: %d\n", *size);
   }
 }


Overwriting seq_block.cu


In [5]:
%%writefile seq_block_2.c

/*
 * Sequential version of matrix multiplication.
 */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h> // just for timing
#include <time.h>

// function declarations
void fillMatrix(int size, float * A);
void MatrixMult(int size, float * __restrict__ A,
                   float * __restrict__ B, float * __restrict__ C);//
void getArguments(int argc, char **argv, int *size, int *verbose);
void debugPrintMatrix(int verbose, int size, float *matrix, const char *msg);
void showMatrix(int size, float * matrix);
void verfiyCorrect(int size, float *matrix);

int main (int argc, char **argv) {

   // default values
   int size = 256;          // num rows, cols of square matrix
   int verbose = 0;         // default to not printing matrices
   getArguments(argc, argv, &size, &verbose); //change defaults

   printf("matrix rows, cols = %d\n", size);

   float * A;  // input matrix
   float * B;  // input matrix
   float * C;  // output matrix

// Use a 'flattened' 1D array of contiguous memory for the matrices
// size = number of rows = number of columns in the square matrices
   size_t num_elements = size * size * sizeof(float);
   A = (float *)malloc(num_elements);
   B = (float *)malloc(num_elements);
   C = (float *)malloc(num_elements);

   fillMatrix(size, A);
   fillMatrix(size, B);
   char msgA[32] = "matrix A after filling:";
   debugPrintMatrix(verbose, size, A, msgA);

   double startTime = omp_get_wtime();

   MatrixMult(size, A, B, C);

   char msgC[32] = "matrix C after MatrixMult(): ";
   debugPrintMatrix(verbose, size, C, msgC);

   double endTime = omp_get_wtime();

   printf("\nTotal omp runtime %f seconds (%f milliseconds)\n",
   (endTime-startTime), (endTime-startTime)*1000);

   verfiyCorrect(size, C);

   free(A); free(B); free(C);
   return 0;
}
////////////////////////////////////// end main

// fill a given square matrix with rows of float values
// equal to each row number
void fillMatrix(int size, float * A) {
   for (int i = 0; i < size; ++i) {
      for (int j = 0; j < size; ++j) {
        A[i*size + j] = ((float)i);
      }
   }
}

// mutiply matrix A times matrix B, placing result in matrix C
void MatrixMult(int size, float * __restrict__ A,
               float * __restrict__ B, float * __restrict__ C) {

   for (int i = 0; i < size; ++i) {
     for (int j = 0; j < size; ++j) {
       float tmp = 0.;
       for (int k = 0; k < size; ++k) {
          tmp += A[i*size + k] * B[k*size + i];
       }
       C[i*size + j] = tmp;    // update cell of C once
     }
   }
}

void getArguments(int argc, char **argv, int *size, int *verbose) {
   // 2 arguments optional:
   //   size of one side of square matrix
   //   verbose printing for debugging
   if (argc > 3) {
      fprintf(stderr,"Use: %s [size] [verbose]\n", argv[0]);
      exit(EXIT_FAILURE);
   }

   if (argc >= 2) {
      *size = atoi(argv[1]);
      if (argc == 3) {
         *verbose = atoi(argv[2]);
      }
   }

   if (*verbose) {
      printf("size of matrix side: %d\n", *size);
   }
}

void debugPrintMatrix(int verbose, int size, float *matrix, const char *msg) {
   if (verbose){
      printf("%s \n", msg);
      showMatrix(size, matrix);
   }
}

// display a given square matrix for debugging purposes
void showMatrix(int size, float * matrix) {
   int i, j;
   for (i=0; i<size; i++){
      for (j=0; j<size; j++) {
         printf("element [%d][%d] = %f \n",i,j, matrix[i*size + j]);
      }
   }
}

// Check whether last row of result matrix is what we expect
void verfiyCorrect(int size, float *matrix) {
   // determine what the last row should contain
   float lastRowValue = 0.0;
   float maxError = 0.0;
   float nextVal = 0.0;

   for (int i=0; i<size; i++)
      lastRowValue += i * (size-1);

   for (int j=0; j<size; j++) {
      nextVal = matrix[(size-1)*size + j];
      maxError = fmaxf(maxError, fabs(nextVal - lastRowValue));
   }
   printf("max error of last row matrix C values: %f\n", maxError);
}

Writing seq_block_2.c


In [7]:
%%shell
g++ seq_block_2.c -fopenmp -o seq_block_2

In [8]:
%%shell
./seq_block_2

matrix rows, cols = 256

Total omp runtime 0.071674 seconds (71.674244 milliseconds)
max error of last row matrix C values: 0.000000
